# M3 CLV-conditioned candidate-item weights on common support — Dunnhumby seed 42

M1의 이진 구매그래프는 보존합니다. pooled 일반관계로 사용자별 후보상품 100개를 먼저 고정하고, 일반·실제 historical CLV proxy·degree-matched CLV shuffle가 동일한 후보 안에서 관계가중치만 다르게 배분합니다. 기존의 양의 초과분 절단은 사용하지 않으며 final test와 holdout은 구성하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib, shutil, subprocess, sys

REVIEWED_SHA = '4434a5d0ff63540c0cd2ca9fe02066d5c20b9dc1'
repo = Path('/content/clv-m2-lightgcn-runner')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
subprocess.run(['git', '-C', str(repo), 'checkout', '-q', REVIEWED_SHA], check=True)
actual_sha = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True).strip()
assert actual_sha == REVIEWED_SHA, (actual_sha, REVIEWED_SHA)
for name in list(sys.modules):
    if name.startswith(('lightgcn_clv', 'clv_m3', 'clv_run_state')):
        del sys.modules[name]
importlib.invalidate_caches()
%cd /content/clv-m2-lightgcn-runner
print('Pinned execution source:', actual_sha)

In [ ]:
import importlib, json, torch
import lightgcn_clv_m3_clv_conditioned_candidate_item as candidate_runner
candidate_runner = importlib.reload(candidate_runner)
assert candidate_runner.COMMON_SUPPORT_CODE_VERSION == 'm3-clv-conditioned-candidate-item-common-support-historical-screen-v2'
assert str(Path(candidate_runner.__file__).resolve()).startswith(str(repo.resolve()))

cfg = candidate_runner.configure_clv_candidate_item_common_support_run()
assert torch.cuda.is_available(), 'Colab 런타임에서 GPU를 선택한 뒤 다시 실행하세요.'
summary = candidate_runner.preflight_summary(cfg)
assert summary['seed'] == 42
assert summary['trained_models'][0] == 'm1_baseline'
assert summary['trained_comparator'] == 'm1_baseline'
assert summary['reused_comparator'] is None
assert summary['historical_development_split']['train_end_inclusive'] == 690
assert summary['historical_development_split']['evaluation_start_inclusive'] == 691
assert summary['historical_development_split']['evaluation_end_inclusive'] == 697
assert summary['historical_development_split']['final_test_constructed'] is False
assert summary['historical_development_split']['holdout_constructed'] is False
assert summary['m3']['historical_clv_proxy'] == 'N_hat * V_hat'
assert summary['m3']['candidate_train_pairs_excluded'] is True
assert summary['m3']['cross_fit_folds'] == 5
assert summary['m3']['item_minimum_distinct_user_support'] == 5
assert summary['m3']['max_candidate_items_per_user'] == 100
assert summary['m3']['candidate_support'] == 'pooled top candidates fixed identically across all arms'
assert summary['m3']['positive_excess_clipping'] is False
assert summary['m3']['gamma'] == 0.075
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['sample_weighting'] is False
assert summary['fixed']['one_training_loop_and_optimizer'] is True
assert summary['fixed']['min_item_interactions'] == 1
assert summary['reading_rule']['accuracy_guardrails'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))

In [ ]:
result_df = candidate_runner.run_clv_candidate_item_common_support_screen(cfg)

In [ ]:
from IPython.display import display

columns = [
    'model_id', 'role',
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'price_purchase_amount_weighted_hit@20',
    'price_purchase_amount_weighted_hit@50',
    'mean_recommended_price_percentile@10',
    'coverage@10', 'n_distinct@10', 'exposure_entropy@10',
    'eff_catalog@10', 'top10_share@10', 'top100_share@10',
]
available = [column for column in columns if column in result_df.columns]
display(result_df[available])

graph = result_df.attrs['graph_diagnostics']
assert graph['common_support']['exact_common_edge_support'] is True
edge_counts = {arm: values['n_edges'] for arm, values in graph['arms'].items()}
assert len(set(edge_counts.values())) == 1, edge_counts
assert all(values['max_active_row_mass_error'] < 1e-6 for values in graph['arms'].values())
print('\n실제 CLV 귀속 판정:')
print(json.dumps(result_df.attrs['attribution_reading'], ensure_ascii=False, indent=2))
print('\n공통 후보지지집 불변조건:', {'edge_counts': edge_counts, 'exact_common_edge_support': True})
print('\nCLV 조건부 후보상품 가중치 그래프 진단:')
print(json.dumps(graph, ensure_ascii=False, indent=2))
print('\n결과 파일:', result_df.attrs['result_paths'])